<div align='center'>
<h1>Sistem Pendukung Rekomendasi Restoran Berbasis Fuzzy Logic</h1>
<h3>Studi Perbandingan Metode Mamdani dan Sugeno</h3>
<br>
<b>Dataset:</b> <a href='https://www.kaggle.com/datasets/shrutimehta/zomato-restaurants-data'>Zomato Restaurants Data</a>
</div>

##### Identitas Kelompok
##### Mata Kuliah : Dasar Kecerdasan Artificial

## 1. Persiapan Library dan Modul

Tahap ini menyiapkan seluruh dependency yang dipakai sepanjang notebook, termasuk modul fuzzy logic dan modul machine learning yang dibangun secara *from scratch*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import sys
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

print('Library standar berhasil dimuat.')

## 1.1 Implementasi Modul Custom From Scratch

Bagian ini berisi penggabungan kode dari file `membership.py`, `mamdani.py`, `sugeno.py`, dan `random_forest.py` agar dapat berjalan langsung di dalam notebook tanpa dependency file eksternal.

In [ ]:
# =============================================================================
# CODE FROM: membership.py
# =============================================================================

def trapezoid(x, a, b, c, d):
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x)
    mask1 = (x >= a) & (x < b)
    if b != a:
        y[mask1] = (x[mask1] - a) / (b - a)
    y[(x >= b) & (x <= c)] = 1.0
    mask3 = (x > c) & (x <= d)
    if d != c:
        y[mask3] = (d - x[mask3]) / (d - c)
    return y

def triangle(x, a, b, c):
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x)
    mask1 = (x >= a) & (x <= b)
    if b != a:
        y[mask1] = (x[mask1] - a) / (b - a)
    mask2 = (x > b) & (x <= c)
    if c != b:
        y[mask2] = (c - x[mask2]) / (c - b)
    y[x == b] = 1.0
    return y

def interp_membership(universe, mf_values, x):
    return float(np.interp(x, universe, mf_values))

def _load_and_preprocess(csv_path='zomato.csv'):
    df_raw = pd.read_csv(csv_path, encoding='latin-1')
    cols = ['Restaurant Name', 'Average Cost for two', 'Price range',
            'Votes', 'Has Table booking', 'Has Online delivery', 'Aggregate rating']
    data_res = df_raw[cols].copy()
    data_res = data_res[data_res['Aggregate rating'] > 0]
    data_res['Has Table booking']   = data_res['Has Table booking'].map({'Yes': 1, 'No': 0})
    data_res['Has Online delivery'] = data_res['Has Online delivery'].map({'Yes': 1, 'No': 0})
    data_res = data_res.dropna().reset_index(drop=True)
    return data_res

def _get_breakpoints(series):
    return {
        'min': float(series.min()),
        'p25': float(series.quantile(0.25)),
        'p50': float(series.quantile(0.50)),
        'p75': float(series.quantile(0.75)),
        'max': float(series.max()),
    }

_data      = _load_and_preprocess()
BP_COST    = _get_breakpoints(_data['Average Cost for two'])
BP_VOTES   = _get_breakpoints(_data['Votes'])
MAX_COST   = BP_COST['max']
MAX_VOTES  = BP_VOTES['max']

U_COST   = np.arange(0, MAX_COST + 10, 10, dtype=float)
U_VOTES  = np.arange(0, MAX_VOTES + 2, 1,  dtype=float)
U_PRICE  = np.arange(1, 6, 1, dtype=float)
U_BINARY = np.array([0.0, 1.0])
U_OUTPUT = np.arange(0, 101, 1, dtype=float)

MF_HARGA = {
    'murah' : trapezoid(U_COST,  BP_COST['min'], BP_COST['min'], BP_COST['p25'], BP_COST['p50']),
    'sedang': triangle (U_COST,  BP_COST['p25'], BP_COST['p50'], BP_COST['p75']),
    'mahal' : trapezoid(U_COST,  BP_COST['p50'], BP_COST['p75'], BP_COST['max'], BP_COST['max']),
}

MF_VOTES = {
    'sedikit': trapezoid(U_VOTES, BP_VOTES['min'], BP_VOTES['min'], BP_VOTES['p25'], BP_VOTES['p50']),
    'sedang' : triangle (U_VOTES, BP_VOTES['p25'], BP_VOTES['p50'], BP_VOTES['p75']),
    'banyak' : trapezoid(U_VOTES, BP_VOTES['p50'], BP_VOTES['p75'], BP_VOTES['max'], BP_VOTES['max']),
}

MF_PRICE = {
    'rendah': trapezoid(U_PRICE, 1, 1, 1, 2),
    'sedang': triangle (U_PRICE, 1, 2, 3),
    'tinggi': trapezoid(U_PRICE, 2, 3, 4, 4),
}

MF_BINARY = {
    'tidak': triangle(U_BINARY, 0, 0, 1),
    'ya'   : triangle(U_BINARY, 0, 1, 1),
}

mf_output = {
    'tidak_layak' : trapezoid(U_OUTPUT, 0,  0,  30, 50),
    'cukup_layak' : triangle (U_OUTPUT, 30, 50, 70),
    'sangat_layak': trapezoid(U_OUTPUT, 50, 70, 100, 100),
}

SUGENO_CONSTANTS = {
    'tidak_layak' : 20,
    'cukup_layak' : 50,
    'sangat_layak': 85,
}

def fuzzify_all(harga, votes, price_range, booking, delivery):
    return {
        'harga'   : {k: interp_membership(U_COST,   v, harga)       for k, v in MF_HARGA.items()},
        'votes'   : {k: interp_membership(U_VOTES,  v, votes)       for k, v in MF_VOTES.items()},
        'price'   : {k: interp_membership(U_PRICE,  v, price_range) for k, v in MF_PRICE.items()},
        'booking' : {k: interp_membership(U_BINARY, v, booking)     for k, v in MF_BINARY.items()},
        'delivery': {k: interp_membership(U_BINARY, v, delivery)    for k, v in MF_BINARY.items()},
    }

def get_mf_curves():
    return {
        'harga': {'x': U_COST, 'xlabel': 'Harga (Average Cost for two)', **MF_HARGA},
        'votes': {'x': U_VOTES, 'xlabel': 'Jumlah Votes', **MF_VOTES},
        'price_range': {'x': U_PRICE, 'xlabel': 'Price Range (1-4)', **MF_PRICE},
        'output': {'x': U_OUTPUT, 'xlabel': 'Skor Rekomendasi (0-100)', **mf_output},
    }

# =============================================================================
# CODE FROM: mamdani.py
# =============================================================================

RULES = [
    {'conditions': [('harga','murah'),  ('votes','banyak')],                          'output': 'sangat_layak'},
    {'conditions': [('harga','murah'),  ('votes','sedang')],                          'output': 'cukup_layak'},
    {'conditions': [('harga','murah'),  ('votes','sedikit')],                         'output': 'cukup_layak'},
    {'conditions': [('harga','sedang'), ('votes','banyak')],                          'output': 'sangat_layak'},
    {'conditions': [('harga','sedang'), ('votes','sedang')],                          'output': 'cukup_layak'},
    {'conditions': [('harga','sedang'), ('votes','sedikit')],                         'output': 'tidak_layak'},
    {'conditions': [('harga','mahal'),  ('votes','banyak')],                          'output': 'cukup_layak'},
    {'conditions': [('harga','mahal'),  ('votes','sedang')],                          'output': 'tidak_layak'},
    {'conditions': [('harga','mahal'),  ('votes','sedikit')],                         'output': 'tidak_layak'},
    {'conditions': [('price','rendah'), ('votes','banyak')],                          'output': 'sangat_layak'},
    {'conditions': [('price','tinggi'), ('votes','sedikit')],                         'output': 'tidak_layak'},
    {'conditions': [('price','sedang'), ('votes','sedang')],                          'output': 'cukup_layak'},
    {'conditions': [('booking','ya'),   ('delivery','ya')],                           'output': 'sangat_layak'},
    {'conditions': [('booking','tidak'),('delivery','tidak'), ('votes','sedikit')],   'output': 'tidak_layak'},
    {'conditions': [('booking','ya'),   ('votes','sedang')],                          'output': 'cukup_layak'},
    {'conditions': [('delivery','ya'),  ('harga','murah')],                           'output': 'sangat_layak'},
    {'conditions': [('booking','tidak'),('harga','mahal')],                           'output': 'tidak_layak'},
]

def get_rekomendasi_level(score):
    if score < 35:
        return 'TIDAK LAYAK'
    elif score < 65:
        return 'CUKUP LAYAK'
    else:
        return 'SANGAT LAYAK'

def evaluate_rules(fz):
    fired = []
    alpha_per_label = {k: 0.0 for k in mf_output}
    for i, rule in enumerate(RULES):
        strengths = []
        for var, lbl in rule['conditions']:
            strengths.append(fz[var][lbl] if lbl in fz.get(var, {}) else 0.0)
        alpha = min(strengths)
        if alpha > 0:
            fired.append({
                'rule_idx'       : i + 1,
                'conditions'     : rule['conditions'],
                'output'         : rule['output'],
                'firing_strength': round(alpha, 4),
            })
            alpha_per_label[rule['output']] = max(alpha_per_label[rule['output']], alpha)
    return fired, alpha_per_label

def aggregate(alpha_per_label):
    agg = np.zeros_like(U_OUTPUT)
    for label, alpha in alpha_per_label.items():
        clipped = np.minimum(alpha, mf_output[label])
        agg = np.maximum(agg, clipped)
    return agg

def centroid(agg):
    den = np.sum(agg)
    return float(np.sum(U_OUTPUT * agg) / den) if den > 0 else 50.0

def mamdani_predict(harga, votes, price_range, booking, delivery):
    t0  = time.time()
    fz  = fuzzify_all(harga, votes, price_range, booking, delivery)
    fired, alpha_per_label = evaluate_rules(fz)
    agg   = aggregate(alpha_per_label)
    score = round(centroid(agg), 4)
    return {
        'score'      : score,
        'level'      : get_rekomendasi_level(score),
        'fired_rules': fired,
        'x_values'   : U_OUTPUT,
        'aggregated' : agg,
        'runtime'    : round(time.time() - t0, 5),
    }

def mamdani_batch(df_input, sample_size=None):
    df_s = df_input.sample(n=sample_size, random_state=42) if sample_size else df_input
    actual, pred = [], []
    for _, row in df_s.iterrows():
        r = mamdani_predict(
            row['Average Cost for two'],
            row['Votes'],
            row['Price range'],
            row['Has Table booking'],
            row['Has Online delivery'],
        )
        actual.append(row['Aggregate rating'] * 20)
        pred.append(r['score'])
    return np.array(actual), np.array(pred)

# =============================================================================
# CODE FROM: sugeno.py
# =============================================================================

def weighted_average(fired_rules):
    num = sum(r['firing_strength'] * r['output_value'] for r in fired_rules)
    den = sum(r['firing_strength'] for r in fired_rules)
    return num / den if den > 0 else 50.0

def sugeno_predict(harga, votes, price_range, booking, delivery):
    t0 = time.time()
    fz = fuzzify_all(harga, votes, price_range, booking, delivery)
    fired = []
    for i, rule in enumerate(RULES):
        strengths = [fz[var][lbl] for var, lbl in rule['conditions'] if lbl in fz.get(var, {})]
        alpha = min(strengths) if strengths else 0.0
        if alpha > 0:
            z = SUGENO_CONSTANTS[rule['output']]
            fired.append({
                'rule_idx'       : i + 1,
                'conditions'     : rule['conditions'],
                'output'         : rule['output'],
                'firing_strength': round(alpha, 4),
                'output_value'   : z,
            })
    score = round(weighted_average(fired), 4) if fired else 50.0
    return {
        'score'      : score,
        'level'      : get_rekomendasi_level(score),
        'fired_rules': fired,
        'runtime'    : round(time.time() - t0, 5),
    }

def sugeno_batch(df_input, sample_size=None):
    df_s = df_input.sample(n=sample_size, random_state=42) if sample_size else df_input
    actual, pred = [], []
    for _, row in df_s.iterrows():
        r = sugeno_predict(
            row['Average Cost for two'],
            row['Votes'],
            row['Price range'],
            row['Has Table booking'],
            row['Has Online delivery'],
        )
        actual.append(row['Aggregate rating'] * 20)
        pred.append(r['score'])
    return np.array(actual), np.array(pred)

# =============================================================================
# CODE FROM: random_forest.py
# =============================================================================

def generate_fuzzy_features(df_in, sample_size=None):
    df_s = df_in.sample(n=sample_size, random_state=42).copy() if sample_size else df_in.copy()
    mamdani_scores, sugeno_scores = [], []
    for _, row in df_s.iterrows():
        mamdani_scores.append(
            mamdani_predict(row['Average Cost for two'], row['Votes'],
                            row['Price range'], row['Has Table booking'],
                            row['Has Online delivery'])['score']
        )
        sugeno_scores.append(
            sugeno_predict(row['Average Cost for two'], row['Votes'],
                           row['Price range'], row['Has Table booking'],
                           row['Has Online delivery'])['score']
        )
    df_s['fuzzy_mamdani'] = mamdani_scores
    df_s['fuzzy_sugeno']  = sugeno_scores
    return df_s

def train_and_evaluate(df_fuzz):
    base_features  = ['Average Cost for two', 'Votes', 'Price range',
                      'Has Table booking', 'Has Online delivery']
    fuzzy_features = base_features + ['fuzzy_mamdani', 'fuzzy_sugeno']
    target = 'Aggregate rating'
    df_fuzz = df_fuzz.dropna(subset=[target])
    results = {}
    for tag, features, label in [
        ('rf_raw',   base_features,  'RF tanpa Fuzzy'),
        ('rf_fuzzy', fuzzy_features, 'RF + Fuzzy Features'),
    ]:
        X = df_fuzz[features].values
        y = df_fuzz[target].values
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        t0  = time.time()
        clf = RandomForestRegressor(n_estimators=100, random_state=42)
        clf.fit(X_train, y_train)
        rt  = round(time.time() - t0, 3)
        pred = clf.predict(X_test)
        mae  = round(float(np.mean(np.abs(y_test - pred))), 4)
        mse  = round(float(np.mean((y_test - pred) ** 2)), 4)
        rmse = round(float(np.sqrt(mse)), 4)
        results[tag] = {
            'label' : label, 'mae'   : mae, 'mse'   : mse, 'rmse'  : rmse,
            'runtime': rt, 'pred'  : pred, 'actual': y_test,
        }
    return results

print('Seluruh modul custom (fuzzy & ml) berhasil disatukan local.')

## 2. Pemuatan Dataset

Dataset diambil dari berkas `zomato.csv` berisi data restoran beserta atribut harga, jumlah votes, dan rating.

In [ ]:
df = pd.read_csv('zomato.csv', encoding='latin-1')
print(f'Jumlah baris x kolom: {df.shape}')
df.head(10)

In [ ]:
# ringkasan struktur dan statistik dataset
print('Informasi struktur dataset:')
df.info()

print('\nStatistik deskriptif:')
df.describe()

## 3. Eksplorasi Data (EDA)

Eksplorasi dilakukan untuk memahami kualitas data, melihat sebaran tiap variabel, dan mendeteksi outlier sebelum data dipakai pada sistem fuzzy.

In [ ]:
# pengecekan kualitas data: missing value & duplikat
print('Missing values per kolom:')
print(df.isnull().sum())
print(f'\nTotal missing value : {df.isnull().sum().sum()}')
print(f'Jumlah baris duplikat: {df.duplicated().sum()}')

In [ ]:
# sebaran variabel numerik (histogram)
num_cols = ['Average Cost for two', 'Votes', 'Price range', 'Aggregate rating']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Distribusi Variabel Numerik', fontsize=14, fontweight='bold')

for i, col in enumerate(num_cols):
    ax = axes[i // 2][i % 2]
    ax.hist(df[col].dropna(), bins=15, color='#2A9D8F', alpha=0.8, edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel('Nilai')
    ax.set_ylabel('Frekuensi')
    ax.axvline(df[col].mean(), color='#073B4C', linestyle='--', linewidth=1.5,
               label=f'Mean: {df[col].mean():.2f}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# deteksi outlier per variabel numerik (boxplot)
fig, axes = plt.subplots(1, 4, figsize=(14, 5))
fig.suptitle('Boxplot Variabel Numerik', fontsize=14, fontweight='bold')

for i, col in enumerate(num_cols):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='#E0F7F4', color='#2A9D8F'),
                    medianprops=dict(color='#1B4965', linewidth=2),
                    whiskerprops=dict(color='#2A9D8F'),
                    capprops=dict(color='#2A9D8F'),
                    flierprops=dict(marker='o', color='#2A9D8F', alpha=0.5))
    axes[i].set_title(col, fontsize=10)
    axes[i].set_ylabel('Nilai')

plt.tight_layout()
plt.show()

In [ ]:
# proporsi variabel kategorikal (pie chart)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.suptitle('Distribusi Variabel Kategorikal', fontsize=14, fontweight='bold')

tb_counts = df['Has Table booking'].value_counts()
axes[0].pie(tb_counts.values, labels=tb_counts.index, autopct='%1.1f%%',
            colors=['#2A9D8F', '#9FD8CB'])
axes[0].set_title('Has Table Booking')

od_counts = df['Has Online delivery'].value_counts()
axes[1].pie(od_counts.values, labels=od_counts.index, autopct='%1.1f%%',
            colors=['#2A9D8F', '#9FD8CB'])
axes[1].set_title('Has Online Delivery')

plt.tight_layout()
plt.show()

In [ ]:
# korelasi antar variabel numerik (heatmap)
df_corr = df[['Average Cost for two', 'Votes', 'Price range', 'Aggregate rating']].copy()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(df_corr.corr(), annot=True, fmt='.2f', cmap='GnBu',
            ax=ax, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Heatmap Korelasi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# hubungan votes & harga terhadap rating (scatter plot)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(df['Votes'], df['Aggregate rating'],
                alpha=0.3, color='#2A9D8F', s=5)
axes[0].set_xlabel('Votes')
axes[0].set_ylabel('Aggregate Rating')
axes[0].set_title('Votes vs Aggregate Rating')

axes[1].scatter(df['Average Cost for two'], df['Aggregate rating'],
                alpha=0.3, color='#1B4965', s=5)
axes[1].set_xlabel('Average Cost for two')
axes[1].set_ylabel('Aggregate Rating')
axes[1].set_title('Average Cost vs Aggregate Rating')

plt.tight_layout()
plt.show()

## 4. Praproses Data

Kolom yang relevan dipilih, baris dengan rating 0 (belum punya rating) dibuang, dan kolom kategorikal Yes/No diubah menjadi biner agar siap dipakai pada tahap fuzzifikasi.

In [ ]:
cols = ['Restaurant Name', 'Average Cost for two', 'Price range',
        'Votes', 'Has Table booking', 'Has Online delivery', 'Aggregate rating']
data = df[cols].copy()

# buang baris rating 0 & encode kolom Yes/No menjadi 1/0
data = data[data['Aggregate rating'] > 0]
data['Has Table booking']   = data['Has Table booking'].map({'Yes': 1, 'No': 0})
data['Has Online delivery'] = data['Has Online delivery'].map({'Yes': 1, 'No': 0})
data = data.dropna().reset_index(drop=True)

print(f'Shape setelah praproses : {data.shape}')
print(f'Sisa missing value      : {data.isnull().sum().sum()}')
data.head()

## 5. Fungsi Keanggotaan (Membership Function)

Fungsi keanggotaan diimplementasikan **from scratch** tanpa library fuzzy, menggunakan dua bentuk kurva: `trapezoid` dan `triangle`.
Titik potong (breakpoint) tiap variabel ditentukan secara **adaptif** mengikuti distribusi data (kuartil p25, p50, p75), bukan nilai tetap.

In [ ]:
print('Breakpoint Harga (Average Cost for two):')
for k, v in BP_COST.items():
    print(f'  {k}: {v}')
print()
print('Breakpoint Votes:')
for k, v in BP_VOTES.items():
    print(f'  {k}: {v}')

In [ ]:
# kurva fungsi keanggotaan untuk seluruh variabel input & output
curves = get_mf_curves()

labels_map = {
    'harga'      : (['murah', 'sedang', 'mahal'],           'Harga (Average Cost for two)'),
    'votes'      : (['sedikit', 'sedang', 'banyak'],        'Jumlah Votes'),
    'price_range': (['rendah', 'sedang', 'tinggi'],         'Price Range (1-4)'),
    'output'     : (['tidak_layak', 'cukup_layak', 'sangat_layak'], 'Skor Rekomendasi (0-100)'),
}

colors = ['#9FD8CB', '#2A9D8F', '#073B4C', '#1B4965']

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Membership Function — Trapezoid & Triangle (From Scratch)', fontsize=14, fontweight='bold')

for idx, (key, (labels, title)) in enumerate(labels_map.items()):
    ax = axes[idx // 2][idx % 2]
    cv = curves[key]
    for i, lbl in enumerate(labels):
        ax.plot(cv['x'], cv[lbl], color=colors[i], linewidth=2.5, label=lbl.capitalize())
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(cv['xlabel'])
    ax.set_ylabel('Derajat Keanggotaan')
    ax.set_ylim(-0.05, 1.15)
    ax.legend(fontsize=9)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## 6. Basis Aturan Fuzzy (17 Rule)

Basis aturan berbentuk IF-THEN yang menjadi inti proses inferensi, menghubungkan kombinasi variabel input dengan tingkat kelayakan rekomendasi.

In [ ]:
print(f"{'No':<4} {'Kondisi':<60} {'Output'}")
print('-' * 80)
for i, rule in enumerate(RULES):
    conds = ' AND '.join([f'{var}={lbl}' for var, lbl in rule['conditions']])
    print(f"{i+1:<4} {conds:<60} {rule['output']}")

## 7. Contoh Kasus Fuzzifikasi

Sebelum masuk ke inferensi penuh, satu contoh data restoran dipakai untuk menunjukkan proses fuzzifikasi nilai input menjadi derajat keanggotaan.

In [ ]:
# contoh input restoran
harga       = 500.0
votes       = 200.0
price_range = 2
booking     = 1      # 1 = Ya
delivery    = 1      # 1 = Ya

In [ ]:
fuzz = fuzzify_all(harga, votes, price_range, booking, delivery)

print('Hasil fuzzifikasi:')
print(f'  Harga          : {fuzz["harga"]}')
print(f'  Votes          : {fuzz["votes"]}')
print(f'  Price Range    : {fuzz["price"]}')
print(f'  Booking        : {fuzz["booking"]}')
print(f'  Delivery       : {fuzz["delivery"]}')

## 8. Inferensi Fuzzy Mamdani

Alur: **Fuzzifikasi → Inferensi (AND=min, agregasi=max) → Defuzzifikasi (Centroid)**

In [ ]:
result_m = mamdani_predict(harga, votes, price_range, booking, delivery)

print(f'Skor Hasil Prediksi : {result_m["score"]}')
print(f'Level Rekomendasi   : {result_m["level"]}')
print(f'Rule aktif          : {len(result_m["fired_rules"])} dari 17')
print(f'Runtime             : {result_m["runtime"]}s')
print()
print('Rule yang aktif:')
for r in result_m['fired_rules']:
    conds = ' AND '.join([f'{v}={l}' for v, l in r['conditions']])
    print(f'  Rule {r["rule_idx"]:>2}: {conds} → {r["output"]} (\u03b1={r["firing_strength"]})')

In [ ]:
# proses agregasi & defuzzifikasi Mamdani
fig, ax = plt.subplots(figsize=(10, 4))

x_vals = result_m['x_values']
agg    = result_m['aggregated']

ax.fill_between(x_vals, agg, alpha=0.4, color='#2A9D8F', label='Area agregasi')
ax.plot(x_vals, agg, color='#1B4965', linewidth=2)
ax.axvline(result_m['score'], color='#073B4C', linestyle='--', linewidth=2,
           label=f'Centroid = {result_m["score"]}')
ax.set_title('Defuzzifikasi Mamdani — Centroid (Center of Gravity)', fontweight='bold')
ax.set_xlabel('Skor Rekomendasi (0-100)')
ax.set_ylabel('Derajat Keanggotaan')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 9. Inferensi Fuzzy Sugeno

Alur: **Fuzzifikasi → Inferensi (AND=min) → Defuzzifikasi (Weighted Average)**
Output tiap rule berupa konstanta: tidak_layak=20, cukup_layak=50, sangat_layak=85

In [ ]:
result_s = sugeno_predict(harga, votes, price_range, booking, delivery)

print(f'Skor Hasil Prediksi : {result_s["score"]}')
print(f'Level Rekomendasi   : {result_s["level"]}')
print(f'Rule aktif          : {len(result_s["fired_rules"])} dari 17')
print(f'Runtime             : {result_s["runtime"]}s')
print()
print('Rule yang aktif:')
for r in result_s['fired_rules']:
    conds = ' AND '.join([f'{v}={l}' for v, l in r['conditions']])
    print(f'  Rule {r["rule_idx"]:>2}: {conds} → {r["output"]} (\u03b1={r["firing_strength"]}, z={r["output_value"]})')

In [ ]:
# proses weighted average Sugeno
fired  = result_s['fired_rules']
labels = [r['output'] for r in fired]
alphas = [r['firing_strength'] for r in fired]
zvals  = [r['output_value'] for r in fired]

x_pos = range(len(fired))

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(x_pos, alphas, color='#2A9D8F', alpha=0.8, width=0.5)
ax.axhline(result_s['score'] / 100, color='#073B4C', linestyle='--', linewidth=2,
           label=f'Weighted Average = {result_s["score"]}')

for bar, lbl, z in zip(bars, labels, zvals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{lbl}\nz={z}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x_pos)
ax.set_xticklabels([f'Rule {r["rule_idx"]}' for r in fired])
ax.set_title('Defuzzifikasi Sugeno — Weighted Average', fontweight='bold')
ax.set_ylabel('Firing Strength (\u03b1)')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 10. Komparasi Mamdani vs Sugeno

Kedua metode diuji dengan input yang sama, lalu dengan beberapa skenario tambahan untuk melihat konsistensi selisih hasilnya.

In [ ]:
print('Perbandingan hasil pada input yang sama:')
print(f'  Input  : Harga={harga}, Votes={votes}, Price Range={price_range}, Booking={booking}, Delivery={delivery}')
print()
print(f'  Mamdani : score={result_m["score"]}, level={result_m["level"]}, runtime={result_m["runtime"]}s')
print(f'  Sugeno  : score={result_s["score"]}, level={result_s["level"]}, runtime={result_s["runtime"]}s')
print(f'  Selisih : {abs(result_m["score"] - result_s["score"]):.4f}')

In [ ]:
# uji beberapa kombinasi input berbeda
test_cases = [
    (200,   10,  1, 0, 0, 'Murah, sepi'),
    (500,   60,  2, 1, 0, 'Sedang, cukup votes'),
    (800,  200,  3, 1, 1, 'Agak mahal, ramai'),
    (2000, 5000, 4, 1, 1, 'Mahal, sangat ramai'),
]

print(f"{'Kasus':<25} {'Mamdani':>10} {'Sugeno':>10} {'Selisih':>10} {'Level M':>15}")
print('-' * 75)
for h, v, p, b, d, label in test_cases:
    m = mamdani_predict(h, v, p, b, d)
    s = sugeno_predict(h, v, p, b, d)
    print(f'{label:<25} {m["score"]:>10.4f} {s["score"]:>10.4f} '
          f'{abs(m["score"]-s["score"]):>10.4f} {m["level"]:>15}')

## 11. Evaluasi Model (MAE, MSE, RMSE)

Ground truth memakai `Aggregate rating × 20` (dikonversi ke skala 0–100) agar dapat dibandingkan langsung dengan skor fuzzy.

In [ ]:
SAMPLE = 2000
print(f'Evaluasi pada {SAMPLE} data sampel acak dari {len(data)} data...')
print()

t0 = time.time()
actual_m, pred_m = mamdani_batch(data, sample_size=SAMPLE)
rt_m_batch = round(time.time() - t0, 2)

t0 = time.time()
actual_s, pred_s = sugeno_batch(data, sample_size=SAMPLE)
rt_s_batch = round(time.time() - t0, 2)

mae_m  = round(float(np.mean(np.abs(actual_m - pred_m))), 4)
mse_m  = round(float(np.mean((actual_m - pred_m)**2)), 4)
rmse_m = round(float(np.sqrt(mse_m)), 4)

mae_s  = round(float(np.mean(np.abs(actual_s - pred_s))), 4)
mse_s  = round(float(np.mean((actual_s - pred_s)**2)), 4)
rmse_s = round(float(np.sqrt(mse_s)), 4)

print(f"{'Metrik':<12} {'Mamdani':>12} {'Sugeno':>12}")
print('-' * 38)
print(f"{'MAE':<12} {mae_m:>12.4f} {mae_s:>12.4f}")
print(f"{'MSE':<12} {mse_m:>12.4f} {mse_s:>12.4f}")
print(f"{'RMSE':<12} {rmse_m:>12.4f} {rmse_s:>12.4f}")
print(f"{'Runtime':<12} {rt_m_batch:>11.2f}s {rt_s_batch:>11.2f}s")

In [ ]:
# visual perbandingan metrik evaluasi
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
fig.suptitle('Perbandingan Metrik Evaluasi Mamdani vs Sugeno', fontsize=13, fontweight='bold')

metrics_vis = [('MAE', mae_m, mae_s), ('RMSE', rmse_m, rmse_s), ('Runtime (s)', rt_m_batch, rt_s_batch)]

for ax, (title, vm, vs) in zip(axes, metrics_vis):
    bars = ax.bar(['Mamdani', 'Sugeno'], [vm, vs],
                  color=['#2A9D8F', '#3A86FF'], width=0.45, edgecolor='white')
    for bar, val in zip(bars, [vm, vs]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vm, vs)*0.02,
                str(val), ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(0, max(vm, vs) * 1.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.legend(handles=[
        plt.Rectangle((0,0),1,1, color='#2A9D8F', label='Mamdani'),
        plt.Rectangle((0,0),1,1, color='#3A86FF', label='Sugeno')
    ], fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# sebaran nilai aktual vs prediksi tiap metode
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, actual, pred, label, clr in [
    (axes[0], actual_m, pred_m, 'Mamdani', '#2A9D8F'),
    (axes[1], actual_s, pred_s, 'Sugeno',  '#3A86FF'),
]:
    ax.scatter(actual, pred, alpha=0.3, s=8, color=clr)
    ax.plot([0, 100], [0, 100], 'k--', linewidth=1, label='Perfect prediction')
    ax.set_xlabel('Aktual (Rating × 20)')
    ax.set_ylabel('Prediksi Skor')
    ax.set_title(f'Aktual vs Prediksi — {label}', fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## 12. Integrasi Fuzzy dengan Random Forest

**Arsitektur:** output fuzzy (Mamdani & Sugeno) dijadikan fitur tambahan untuk Random Forest.
Fuzzy tidak menggantikan ML — output fuzzy menjadi informasi tambahan yang memperkaya fitur model.

In [ ]:
print('Menghasilkan fitur fuzzy untuk tiap baris data...')
df_fuzz = generate_fuzzy_features(data, sample_size=2000)
print(f'Selesai. Shape: {df_fuzz.shape}')
df_fuzz[['Restaurant Name', 'Aggregate rating', 'fuzzy_mamdani', 'fuzzy_sugeno']].head(10)

In [ ]:
print('Melatih model Random Forest...')
results = train_and_evaluate(df_fuzz)

print()
print(f"{'Model':<25} {'MAE':>8} {'MSE':>8} {'RMSE':>8} {'Runtime':>10}")
print('-' * 65)
for tag in ['rf_raw', 'rf_fuzzy']:
    r = results[tag]
    print(f"{r['label']:<25} {r['mae']:>8.4f} {r['mse']:>8.4f} {r['rmse']:>8.4f} {r['runtime']:>9.3f}s")

In [ ]:
# perbandingan performa RF dengan & tanpa fitur fuzzy
labels_rf = [results['rf_raw']['label'], results['rf_fuzzy']['label']]
mae_vals   = [results['rf_raw']['mae'],   results['rf_fuzzy']['mae']]
rmse_vals  = [results['rf_raw']['rmse'],  results['rf_fuzzy']['rmse']]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.suptitle('Perbandingan RF tanpa Fuzzy vs RF + Fuzzy Features', fontsize=12, fontweight='bold')

for ax, title, vals in [(axes[0],'MAE',mae_vals),(axes[1],'RMSE',rmse_vals)]:
    bars = ax.bar(labels_rf, vals, color=['#888888','#2A9D8F'], width=0.45, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(vals)*0.02,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(0, max(vals)*1.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

## 13. Ringkasan dan Kesimpulan

In [ ]:
print('=' * 60)
print('KESIMPULAN')
print('=' * 60)
print()
print('Perbandingan Mamdani vs Sugeno:')
print(f'  MAE            - Mamdani: {mae_m:.4f} | Sugeno: {mae_s:.4f}')
print(f'  MSE            - Mamdani: {mse_m:.4f} | Sugeno: {mse_s:.4f}')
print(f'  RMSE           - Mamdani: {rmse_m:.4f} | Sugeno: {rmse_s:.4f}')
print(f'  Runtime batch  - Mamdani: {rt_m_batch}s   | Sugeno: {rt_s_batch}s')
print()
print('Interpretasi:')
winner_acc = 'Mamdani' if mae_m <= mae_s else 'Sugeno'
winner_spd = 'Sugeno'  if rt_s_batch <= rt_m_batch else 'Mamdani'
speedup    = round(rt_m_batch / rt_s_batch, 1) if rt_s_batch > 0 else '-'
print(f'  - {winner_acc} menghasilkan akurasi lebih tinggi (MAE lebih rendah)')
print(f'  - {winner_spd} {speedup}x lebih cepat dari metode lainnya')
print()
print('Terdapat trade-off antara akurasi dan kecepatan komputasi.')
print('Untuk sistem rekomendasi restoran yang mengutamakan presisi,')
print('Mamdani lebih direkomendasikan.')